# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Савков Олександр'
ГРУПА        = 'ШІ-33'
НОМЕР        = 6

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'


In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 6   ·   Савков Олександр   ·   ШІ-33

Ваші вісім стовпців:
   1. ПІП
   2. Примітка
   3. Перебіг
   4. Дата розтину
   5. Ліжко днів
   6. Т вище38
   7. МНВ1
   8. Креат2

Ваша пара для аналізу зв’язку: Ліжко днів та Вік

Ваше додаткове питання:
   Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,Примітка,Перебіг,Дата розтину,Ліжко днів,Т вище38,МНВ1,Креат2
0,Федорюк Петро Юрійович,NaN,важкий,2022-03-11,14,4 дні до госп,3.10,92
1,Загул Григорій Сергійович,"отримував актемра, часті носові кровотечі, втрата нюху",важкий,NaT,22,5 до гос,1.22,73
2,Бондара Лариса Павлівна,часта втрата свідомосі в стаціонарі,важкий,NaT,20,10 до гос,1.03,117.2
3,Ватаманюка Уляна Дмитрівна,панкреонекроз,важкий,NaT,20,9 до гос,1.69,105.1
4,Бабійа Лариса Олегівна,NaN,важкий,NaT,27,NaN,1.15,101
5,Ковалика ганна Андріївна,NaN,важкий,2021-12-26,9(5 у реан),NaN,0.90,172
6,Височан Віктор Володимирович,"глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,NaT,18,7 дні до госп,1.09,97.8
7,Дзюбаа Ірина Іванівна,NaN,серед важ,NaT,10,9 днів до госп,0.94,107


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    object        
 1   Примітка      45 non-null     object        
 2   Перебіг       121 non-null    object        
 3   Дата розтину  61 non-null     datetime64[ns]
 4   Ліжко днів    120 non-null    object        
 5   Т вище38      77 non-null     object        
 6   МНВ1          90 non-null     float64       
 7   Креат2        102 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '20', '27']

--- Т вище38
    непорожніх 77 з 121 | унікаль

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'ПІП': ('ідентифікатор',
            'прізвище та ім’я ідентифікують пацієнта: 121 унікальне значення на 121 рядок, '
            'порівняння двох значень дає лише «той самий хворий чи інший»'),
    'Примітка': ('номінальна',
                 'вільний клінічний коментар: значення можна порівнювати тільки на збіг, '
                 'порядку між «панкреонекроз» і «втрата нюху» не існує'),
    'Перебіг': ('порядкова',
                'три рівні тяжкості впорядковані від «серед важ» до «вкрай важкий», '
                'але відстань між сусідніми рівнями не виміряна і не однакова'),
    'Дата розтину': ('інтервальна',
                     'різниця двох дат має зміст у днях, а нуль календаря умовний, '
                     'тому відношення двох дат не означає нічого'),
    'Ліжко днів': ('відношень',
                   'тривалість госпіталізації має природний нуль і допускає відношення: '
                   '20 ліжко-днів удвічі більше за 10'),
    'Т вище38': ('відношень',
                 'кількість днів із температурою вище 38 має природний нуль і допускає '
                 'відношення, хоча записана текстом і число треба спершу витягти'),
    'МНВ1': ('відношень',
             'міжнародне нормалізоване відношення є часткою двох протромбінових часів, '
             'має природний нуль і допускає твердження «удвічі вище»'),
    'Креат2': ('відношень',
               'концентрація креатиніну в мікромолях на літр має природний нуль, '
               'тому відношення двох значень осмислене'),
    'Вік': ('відношень',
            'вік має природний нуль у момент народження, і твердження «удвічі старший» '
            'має зміст; потрібен для пари четвертого етапу'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df


,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,прізвище та ім’я ідентифікують пацієнта: 121 унікальне з...
1,Примітка,номінальна,вільний клінічний коментар: значення можна порівнювати т...
2,Перебіг,порядкова,три рівні тяжкості впорядковані від «серед важ» до «вкра...
3,Дата розтину,інтервальна,"різниця двох дат має зміст у днях, а нуль календаря умов..."
4,Ліжко днів,відношень,тривалість госпіталізації має природний нуль і допускає ...
5,Т вище38,відношень,кількість днів із температурою вище 38 має природний нул...
6,МНВ1,відношень,міжнародне нормалізоване відношення є часткою двох протр...
7,Креат2,відношень,концентрація креатиніну в мікромолях на літр має природн...


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


Паспорт складено не за типом даних, а за змістом величин. Pandas бачить «Ліжко днів» і
«Креат2» як object лише через одне текстове значення в кожному, а «Перебіг» як рядок — хоча
перше й друге належать шкалі відношень, а третє порядковій. Окремо винесено «Вік»: він не
входить у вісімку варіанта, але потрібен як друга змінна пари четвертого етапу, і без його
шкали вибір міри зв’язку обґрунтувати неможливо.

---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
from scipy import stats

SEVERITY = {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}


def to_number(col):
    s = df[col]
    if col == 'Перебіг':
        return s.str.strip().map(SEVERITY).dropna()
    if col == 'Дата розтину':
        return s.dropna()
    if s.dtype == object:
        s = s.astype(str).str.extract(r'(\d+\.?\d*)')[0]
    return pd.to_numeric(s, errors='coerce').dropna()


def допустима_статистика(стовпець, шкала):
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш == 'ідентифікатор':
        return р

    р['мода'] = з.mode().iloc[0]
    if ш == 'номінальна':
        return р

    v = to_number(стовпець)
    р['медіана'] = v.median()
    р['Q1'] = v.quantile(0.25)
    р['Q3'] = v.quantile(0.75)
    if ш == 'порядкова':
        return р

    р['середнє'] = v.mean()
    р['відхилення'] = v.std()
    if ш == 'інтервальна':
        return р

    р['CV'] = round(float(v.std() / v.mean()), 4)
    р['геом. середнє'] = round(float(stats.gmean(v[v > 0])), 4)
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    for k, v in допустима_статистика(c, ПАСПОРТ[c][0]).items():
        print(f'    {k}: {v}')
    print()


--- ПІП  (ідентифікатор)
    непорожніх: 121
    унікальних: 121

--- Примітка  (номінальна)
    непорожніх: 45
    унікальних: 44
    мода: Поліалергетик

--- Перебіг  (порядкова)
    непорожніх: 121
    унікальних: 3
    мода: важкий
    медіана: 2.0
    Q1: 2.0
    Q3: 2.0

--- Дата розтину  (інтервальна)
    непорожніх: 61
    унікальних: 50
    мода: 2022-06-07 00:00:00
    медіана: 2022-04-23 00:00:00
    Q1: 2022-03-29 00:00:00
    Q3: 2022-05-07 00:00:00
    середнє: 2022-04-06 13:22:37.377049088
    відхилення: 72 days 13:34:38.466302692

--- Ліжко днів  (відношень)
    непорожніх: 120
    унікальних: 40
    мода: 10
    медіана: 15.0
    Q1: 10.0
    Q3: 22.0
    середнє: 17.883333333333333
    відхилення: 12.936940355830204
    CV: 0.7234
    геом. середнє: 14.1555

--- Т вище38  (відношень)
    непорожніх: 77
    унікальних: 28
    мода: 3
    медіана: 4.0
    Q1: 3.0
    Q3: 5.0
    середнє: 4.220779220779221
    відхилення: 1.8540381008928977
    CV: 0.4393
    геом. сере

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

Функція побудована як драбина: кожна шкала повертає все, що дозволяє попередня, і додає
своє. Ідентифікатор отримує лише кількість непорожніх і унікальних значень, номінальна додає
моду, порядкова — медіану й квартилі, інтервальна — середнє й відхилення, відношень —
коефіцієнт варіації та геометричне середнє. Допоміжна `to_number` зводить стовпець до чисел,
придатних для цих статистик: для «Перебігу» це коди порядку, для «Дати розтину» самі дати,
для решти — число, витягнуте з тексту регулярним виразом.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Т вище38,77.0,28.0,3.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
МНВ1,90.0,NaN,NaN,NaN,1.333778,0.8,1.0525,1.19,1.34,10.6,1.032131
Креат2,102.0,72.0,77.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
autopsy = df['Дата розтину'].dropna()
cv_1970 = round(float(autopsy.astype('int64').std() / autopsy.astype('int64').mean()), 4)
days_1900 = (autopsy - pd.Timestamp('1900-01-01')).dt.days
cv_1900 = round(float(days_1900.std() / days_1900.mean()), 4)

fever_raw = pd.to_numeric(df['Т вище38'], errors='coerce')
fever_all = to_number('Т вище38')
filled = int(df['Т вище38'].notna().sum())
lost = filled - int(fever_raw.notna().sum())

severity = df['Перебіг'].str.strip()
mean_123 = round(float(severity.map(SEVERITY).mean()), 4)
mean_1210 = round(float(severity.map({'серед важ': 1, 'важкий': 2, 'вкрай важкий': 10}).mean()), 4)

creat = pd.to_numeric(df['Креат2'], errors='coerce')

БЕЗГЛУЗДІ = [
    ('Дата розтину', 'коефіцієнт варіації', cv_1970,
     f'величина ділить розкид на середнє, а середнє дати залежить від вибору нуля: '
     f'від 1970 року виходить {cv_1970}, від 1900 року — {cv_1900}'),
    ('Т вище38', 'середнє після pd.to_numeric', round(float(fever_raw.mean()), 4),
     f'функція мовчки відкинула {lost} із {filled} заповнених значень, записаних словами; '
     f'справжнє середнє після витягання числа з тексту дорівнює {round(float(fever_all.mean()), 4)}'),
    ('Перебіг', 'середнє порядкових кодів', mean_123,
     f'відстані між рівнями тяжкості невідомі: при кодах 1/2/10 те саме середнє дорівнює '
     f'{mean_1210}, тобто число задає кодувальник, а не хворі'),
    ('Креат2', 'count у describe', int(creat.notna().sum()),
     f'describe показує {int(creat.notna().sum())} спостережень при '
     f'{int(df["Креат2"].notna().sum())} заповнених: рядок «немає» зник при зведенні до числа'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])


,Стовпець,Статистика,Значення,Чому безглузде
0,Дата розтину,коефіцієнт варіації,0.0038,"величина ділить розкид на середнє, а середнє дати залежи..."
1,Т вище38,середнє після pd.to_numeric,3.4444,"функція мовчки відкинула 50 із 77 заповнених значень, за..."
2,Перебіг,середнє порядкових кодів,1.7603,відстані між рівнями тяжкості невідомі: при кодах 1/2/10...
3,Креат2,count у describe,101.0000,describe показує 101 спостережень при 102 заповнених: ря...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Ліжко днів × Вік

Ліжко днів: шкала — відношень
Вік: шкала — відношень

    Ліжко днів  Вік
0           14   55
1           22   43
2           20   54
3           20   68
4           27   65
5  9(5 у реан)   66


In [12]:
ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = ('обидві змінні належать шкалі відношень: і ліжко-дні, і вік мають природний '
                  'нуль та допускають відношення, тому лінійна кореляція Пірсона для цієї пари '
                  'визначена коректно')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

pair = pd.DataFrame({A: to_number(A), B: to_number(B)}).dropna()
r, p_pearson = stats.pearsonr(pair[A], pair[B])
rho, p_spearman = stats.spearmanr(pair[A], pair[B])

print(f'спостережень у парі: {len(pair)} із {len(df)}')
print(f'Пірсон:  r   = {r:+.4f}   p = {p_pearson:.4f}')
print(f'Спірмен: rho = {rho:+.4f}   p = {p_spearman:.4f}')
print()
print(f'{A}: медіана {pair[A].median():g}, максимум {pair[A].max():g}, '
      f'асиметрія {pair[A].skew():.3f}')
print(f'{B}: медіана {pair[B].median():g}, максимум {pair[B].max():g}, '
      f'асиметрія {pair[B].skew():.3f}')


спостережень у парі: 120 із 121
Пірсон:  r   = -0.0996   p = 0.2791
Спірмен: rho = -0.1940   p = 0.0337

Ліжко днів: медіана 15, максимум 79, асиметрія 2.131
Вік: медіана 66, максимум 97, асиметрія -0.148


In [13]:
codings = [('1 / 2 / 3', SEVERITY),
           ('1 / 2 / 10', {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 10}),
           ('3 / 1 / 2', {'серед важ': 3, 'важкий': 1, 'вкрай важкий': 2})]

for name, coding in codings:
    value, pv = stats.pearsonr(severity.map(coding), df['Вік'])
    print(f'Пірсон(Перебіг у кодах {name:>10}, Вік) = {value:+.4f}   p = {pv:.4f}')

rho_sev, p_sev = stats.spearmanr(severity.map(SEVERITY), df['Вік'])
print()
print(f'Спірмен(Перебіг, Вік) = {rho_sev:+.4f}   p = {p_sev:.4f}')


Пірсон(Перебіг у кодах  1 / 2 / 3, Вік) = +0.3541   p = 0.0001
Пірсон(Перебіг у кодах 1 / 2 / 10, Вік) = +0.2237   p = 0.0137
Пірсон(Перебіг у кодах  3 / 1 / 2, Вік) = -0.3482   p = 0.0001

Спірмен(Перебіг, Вік) = +0.3875   p = 0.0000


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

Пара «Ліжко днів» і «Вік» належить до поєднання відношень на відношень, тому за правилом
шкал береться Пірсон. Він дає -0.0996 при p = 0.279, тобто лінійного зв’язку немає. Спірмен на
тих самих даних дає -0.1940 при p = 0.034, і ця розбіжність не випадкова: ліжко-дні мають
важкий правий хвіст, медіана 15 проти максимуму 79, і кілька довгих госпіталізацій тягнуть
лінійну кореляцію до нуля, тоді як рангова їх стискає. Висновок формулюється обережно: слабка
монотонна тенденція до коротшої госпіталізації в старших хворих є, лінійної залежності немає.

Контрприклад показує, чим платить той, хто бере Пірсона без огляду на шкалу. «Перебіг» —
порядкова змінна, і сам коефіцієнт із «Віком» змінюється з +0.3541 на +0.2237, якщо рівні
закодувати 1, 2 і 10 замість 1, 2 і 3, та на -0.3482, якщо порядок переплутати. Жодне з цих
трьох чисел не є властивістю хворих: усі три описують таблицю кодування. Спірмен на тій самій
парі дає +0.3875 і від кодування не залежить взагалі, бо працює з рангами.

In [14]:
ПІДГОТОВКА = {
    'ПІП': 'у модель не беруть: ідентифікатор, який не є величиною',
    'Примітка': 'як текст у модель не беруть; за потреби — бінарні прапорці за ключовими словами',
    'Перебіг': 'впорядкована категорія з явно заданим порядком серед важ < важкий < вкрай важкий',
    'Дата розтину': 'замінити на тривалість у днях від поступлення плюс бінарна ознака наявності розтину',
    'Ліжко днів': 'витягти число з тексту, логарифмувати через правий хвіст, далі стандартизувати',
    'Т вище38': 'витягти число днів і окремо прапорець контексту: до госпіталізації, вдома чи в стаціонарі',
    'МНВ1': 'стандартизація; нуль природний, тому допустима й нормалізація в нуль-один',
    'Креат2': 'замінити текст «немає» на NaN, далі стандартизація',
    'Вік': 'стандартизація; категоризація на вікові групи лише за потреби інтерпретації',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])


,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"у модель не беруть: ідентифікатор, який не є величиною"
1,Примітка,номінальна,як текст у модель не беруть; за потреби — бінарні прапор...
2,Перебіг,порядкова,впорядкована категорія з явно заданим порядком серед важ...
3,Дата розтину,інтервальна,замінити на тривалість у днях від поступлення плюс бінар...
4,Ліжко днів,відношень,"витягти число з тексту, логарифмувати через правий хвіст..."
5,Т вище38,відношень,витягти число днів і окремо прапорець контексту: до госп...
6,МНВ1,відношень,"стандартизація; нуль природний, тому допустима й нормалі..."
7,Креат2,відношень,"замінити текст «немає» на NaN, далі стандартизація"


In [15]:
prep = pd.DataFrame(index=df.index)

order = pd.CategoricalDtype(['серед важ', 'важкий', 'вкрай важкий'], ordered=True)
prep['severity'] = severity.astype(order).cat.codes

admission = pd.to_datetime(df['Поступлення'], errors='coerce')
prep['autopsy_days'] = (df['Дата розтину'] - admission).dt.days
prep['has_autopsy'] = df['Дата розтину'].notna().astype(int)
impossible = prep.index[prep['autopsy_days'] < 0]
prep.loc[impossible, 'autopsy_days'] = np.nan

bed = to_number('Ліжко днів')
bed_log = np.log1p(bed)
prep['bed_log'] = bed_log
prep['bed_z'] = (bed_log - bed_log.mean()) / bed_log.std()

creat_clean = pd.to_numeric(df['Креат2'].replace('немає', np.nan), errors='coerce')
prep['creat_z'] = (creat_clean - creat_clean.mean()) / creat_clean.std()

print(prep.describe().T.round(4).to_string())
print()
print('асиметрія ліжко-днів: %.3f до логарифма, %.3f після' % (bed.skew(), bed_log.skew()))
print('дат розтину раніше за поступлення: %d, вилучено' % len(impossible))
print(df.loc[impossible, ['Поступлення', 'Виписка', 'Дата розтину']].to_string())


              count     mean      std     min     25%      50%      75%      max
severity      121.0   0.7603   0.4477  0.0000  1.0000   1.0000   1.0000   2.0000
autopsy_days   59.0  17.2203  14.6371  3.0000  8.0000  15.0000  21.5000  80.0000
has_autopsy   121.0   0.5041   0.5021  0.0000  0.0000   1.0000   1.0000   1.0000
bed_log       120.0   2.7377   0.6506  0.6931  2.3979   2.7726   3.1355   4.3820
bed_z         120.0  -0.0000   1.0000 -3.1427 -0.5223   0.0536   0.6115   2.5275
creat_z       101.0   0.0000   1.0000 -0.7295 -0.3691  -0.1562   0.1328   8.6490

асиметрія ліжко-днів: 2.131 до логарифма, -0.277 після
дат розтину раніше за поступлення: 2, вилучено
   Поступлення              Виписка Дата розтину
81  2022-03-26  2022-05-02 00:00:00   2022-02-03
90  2021-12-15  2022-01-06 00:00:00   2021-01-06


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


Дві перевірки нижче відповідають на питання до захисту. Перша показує, що
нормалізація сталого стовпця не падає з помилкою, а мовчки дає NaN у кожному рядку, бо знаменник
max мінус min дорівнює нулю; захищений варіант зупиняється одразу й називає причину. Друга
стосується стовпця ЛН, записаного римськими цифрами: усі одиниці в ньому є кириличною літерою
І з кодом 0x406, окрім рівня ІІІ-IV, де IV набрано латиницею з кодами 0x49 і 0x56. Автоматичне
сортування рядків спирається на коди символів, тому на такій мішанці алфавітів воно ненадійне,
і порядок рівнів задається списком вручну.

In [17]:
const = pd.Series([5.0] * len(df))
scaled = (const - const.min()) / (const.max() - const.min())
print('сталий стовпець: унікальних', scaled.nunique(), '| NaN', int(scaled.isna().sum()))


def normalize(s):
    span = s.max() - s.min()
    if span == 0:
        raise ValueError('сталий стовпець: нормалізація не визначена')
    return (s - s.min()) / span


try:
    normalize(const)
except ValueError as error:
    print('захищений варіант:', error)


сталий стовпець: унікальних 0 | NaN 121
захищений варіант: сталий стовпець: нормалізація не визначена


In [18]:
ln = df['ЛН'].dropna().str.strip()
LN_ORDER = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']

for level in LN_ORDER:
    print(f'{level:<8}', [hex(ord(ch)) for ch in level])

print()
print(ln.astype(pd.CategoricalDtype(LN_ORDER, ordered=True))
        .value_counts().reindex(LN_ORDER).to_string())


І        ['0x406']
І-ІІ     ['0x406', '0x2d', '0x406', '0x406']
ІІ       ['0x406', '0x406']
ІІ-ІІІ   ['0x406', '0x406', '0x2d', '0x406', '0x406', '0x406']
ІІІ      ['0x406', '0x406', '0x406']
ІІІ-IV   ['0x406', '0x406', '0x406', '0x2d', '0x49', '0x56']

ЛН
І         12
І-ІІ       9
ІІ        21
ІІ-ІІІ    59
ІІІ       17
ІІІ-IV     3


In [19]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [20]:
ВИПРАВЛЕННЯ = [
    (1, 'середнє номінальної змінної',
     'стовпець помер/жив закодовано 1 і 2 довільно; при кодах 0 і 1 те саме середнє дорівнює '
     '0.4959, при кодах 1 і 10 — 5.4628, тобто число описує таблицю кодування, а не хворих',
     'рахувати частку класу: померли 61 зі 121, тобто 0.5041'),
    (2, 'категорії не зведені до спільного запису',
     'чотири «статі» виникли з двох: ж 56 і жін 19 — одна група, ч 34 і чол 12 — друга; '
     'число 4 вимірює акуратність введення, а не демографію',
     'звести до першої літери: виходить 75 жінок і 46 чоловіків'),
    (3, 'порядок рівнів узято з абетки, і на ньому пораховано середнє',
     'sort_index упорядкував рівні алфавітно, тому найлегшим названо «важкий», а найважчим '
     '«середній»; середнє 5.9 таких кодів не вимірює тяжкості, бо порядкова шкала не має відстаней',
     'задати порядок рівнів вручну після очищення пробілів і описок, далі брати моду й медіану'),
    (4, 'статистика порахована на чверті заповнених значень',
     'стовпець КТ заповнений у 44 хворих, але до числа звелися 27: записи «20-30%» і «до 17%» '
     'мовчки стали NaN, тож 39.6 % — середнє по тих, хто записав обсяг часткою',
     'розібрати діапазони в середину інтервалу або рахувати на 27 значеннях, назвавши обсяг прямо'),
    (5, 'змішані одиниці в одному стовпці',
     'мінімум 0.9 поруч із максимумом 98 означає, що одне значення записане часткою, а не '
     'відсотком, і воно тягне середнє вниз',
     'привести до спільних одиниць: після заміни 0.9 на 90 середнє стає 82.79 замість 82.05'),
    (6, 'коефіцієнт варіації на інтервальній шкалі',
     'величина ділить розкид на середнє, а середнє дати залежить від нуля відліку: від 1970 року '
     'виходить 0.5949 %, від 1900 року 0.2536 %, від 2021 року 30.04 % на тих самих 120 датах',
     'описувати розкид дат розмахом і квартилями: 463 дні, IQR 152 дні, пік 30 госпіталізацій '
     'у квітні 2022 проти однієї в лютому 2021'),
    (7, 'кореляція Пірсона для пари номінальних змінних',
     'ні стать, ні результат лікування не мають порядку, тому вислів «лінійний зв’язок» для них '
     'не визначений, а саме число 0.177 залежить від того, кого позначили нулем',
     'будувати таблицю спряженості й рахувати хі-квадрат із коефіцієнтом Крамера: '
     'chi2 = 3.0861, p = 0.0790, V = 0.1597'),
    (8, 'нормалізація без перевірки варіативності й без огляду на шкалу',
     'у стовпці ФЕРИТ2 одне значення на 121 рядок, max дорівнює min, і ділення на нуль дало '
     'самі NaN без жодної помилки; разом із тим у набір числових потрапив номінальний помер/жив',
     'перевіряти розмах перед діленням і виключати номінальні стовпці зі списку числових'),
    (9, 'dropna на наборі з розрідженими пропусками',
     'повністю заповнені лише 8 стовпців із 77, тому перетин умов порожній: лишилось 0 рядків '
     'зі 121, і «повністю заповнений набір» складається з нуля хворих',
     'працювати постовпцево, прибираючи пропуски лише за тими ознаками, які входять у конкретне '
     'обчислення'),
    (10, 'витік цільової змінної в ознаки',
     'дата розтину заповнена рівно у 61 померлого і в жодного з 60 живих, тому кореляція 1.00 '
     'відтворює ціль, а не передбачає її; на новому хворому цієї ознаки ще не існує',
     'виключати ознаки, що виникають після результату, і будувати рейтинг лише на даних, '
     'відомих до кінця лікування'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])


,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,середнє номінальної змінної,стовпець помер/жив закодовано 1 і 2 довільно; при кодах ...,"рахувати частку класу: померли 61 зі 121, тобто 0.5041"
1,2,категорії не зведені до спільного запису,чотири «статі» виникли з двох: ж 56 і жін 19 — одна груп...,звести до першої літери: виходить 75 жінок і 46 чоловіків
2,3,"порядок рівнів узято з абетки, і на ньому пораховано сер...","sort_index упорядкував рівні алфавітно, тому найлегшим н...",задати порядок рівнів вручну після очищення пробілів і о...
3,4,статистика порахована на чверті заповнених значень,"стовпець КТ заповнений у 44 хворих, але до числа звелися...",розібрати діапазони в середину інтервалу або рахувати на...
4,5,змішані одиниці в одному стовпці,"мінімум 0.9 поруч із максимумом 98 означає, що одне знач...",привести до спільних одиниць: після заміни 0.9 на 90 сер...
5,6,коефіцієнт варіації на інтервальній шкалі,"величина ділить розкид на середнє, а середнє дати залежи...","описувати розкид дат розмахом і квартилями: 463 дні, IQR..."
6,7,кореляція Пірсона для пари номінальних змінних,"ні стать, ні результат лікування не мають порядку, тому ...",будувати таблицю спряженості й рахувати хі-квадрат із ко...
7,8,нормалізація без перевірки варіативності й без огляду на...,"у стовпці ФЕРИТ2 одне значення на 121 рядок, max дорівню...",перевіряти розмах перед діленням і виключати номінальні ...
8,9,dropna на наборі з розрідженими пропусками,"повністю заповнені лише 8 стовпців із 77, тому перетин у...","працювати постовпцево, прибираючи пропуски лише за тими ..."
9,10,витік цільової змінної в ознаки,дата розтину заповнена рівно у 61 померлого і в жодного ...,"виключати ознаки, що виникають після результату, і будув..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

Десять помилок чужого звіту лежать по одній у кожному з десяти розділів, і всі вони
належать до двох типів, названих у методичці. Перший тип — статистика, недопустима для шкали:
середнє номінальних кодів у розділі 1, середнє алфавітно впорядкованих рівнів у розділі 3,
коефіцієнт варіації дати в розділі 6, кореляція Пірсона між двома номінальними змінними в
розділі 7. Другий тип — число, пораховане на непридатних даних: незведені категорії в розділі 2,
втрата сімнадцяти з сорока чотирьох значень у розділі 4, змішані одиниці в розділі 5, ділення на
нуль у сталому стовпці в розділі 8, порожній набір після dropna у розділі 9 і витік цілі через
дату розтину в розділі 10.

Нижче показано правильне обчислення поруч із хибним для чотирьох розділів: першого, шостого,
сьомого й десятого. Решту шість закрито в таблиці вище описом помилки й способом виправлення.

In [21]:
wrong = pd.read_excel('Ковід_дані_навчальні.xlsx', sheet_name='Лист1', engine='openpyxl')
died = (wrong['помер/жив'] == 1).astype(int)

print('РОЗДІЛ 1')
print('  хибно:', round(float(wrong['помер/жив'].mean()), 4), '— середнє номінальних кодів')
for name, coding in (('0 і 1', {1: 0, 2: 1}), ('1 і 10', {1: 1, 2: 10})):
    print(f'    те саме середнє при кодах {name}: {wrong["помер/жив"].map(coding).mean():.4f}')
print('  правильно:', round(float(died.mean()), 4),
      f'— частка померлих, {int(died.sum())} зі {len(wrong)}')

print()
print('РОЗДІЛ 6')
dates = pd.to_datetime(wrong['Поступлення'], errors='coerce').dropna()
for epoch in ('1970-01-01', '1900-01-01', '2021-01-01'):
    x = (dates - pd.Timestamp(epoch)).dt.days
    print(f'  хибно: CV від {epoch} = {100 * x.std() / x.mean():.4f} %')
span = (dates.max() - dates.min()).days
iqr = (dates.quantile(0.75) - dates.quantile(0.25)).days
print(f'  правильно: розмах {span} днів, IQR {iqr} днів, '
      f'std {(dates - dates.min()).dt.days.std():.1f} днів')
print('  по місяцях:', dates.dt.to_period('M').value_counts().sort_index().tail(4).to_dict())

print()
print('РОЗДІЛ 7')
sex = wrong['Стать'].str.strip().str[0].map({'ж': 'жінки', 'ч': 'чоловіки'})
print('  хибно: Пірсон =', round(float(sex.map({'жінки': 1, 'чоловіки': 0}).corr(died)), 4))
table = pd.crosstab(sex, died)
chi2, p_value, dof, _ = stats.chi2_contingency(table)
cramer = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
print(table.to_string())
print(f'  правильно: chi2 = {chi2:.4f}, p = {p_value:.4f}, V Крамера = {cramer:.4f}')

print()
print('РОЗДІЛ 10')
leak = pd.to_datetime(wrong['Дата розтину'], errors='coerce').astype('int64')
print('  хибно: кореляція «Дата розтину» з летальністю =', round(abs(float(leak.corr(died))), 4))
print('    заповнена у померлих:', int(wrong.loc[died == 1, 'Дата розтину'].notna().sum()),
      '| у живих:', int(wrong.loc[died == 0, 'Дата розтину'].notna().sum()))
clean = [c for c in wrong.select_dtypes('number').columns if c != 'помер/жив']
rank = pd.Series({c: abs(wrong[c].corr(died)) for c in clean}).dropna()
print('  правильно, без ознак, що виникають після результату:')
print(rank.sort_values(ascending=False).head(5).round(3).to_string())


РОЗДІЛ 1
  хибно: 1.4959 — середнє номінальних кодів
    те саме середнє при кодах 0 і 1: 0.4959
    те саме середнє при кодах 1 і 10: 5.4628
  правильно: 0.5041 — частка померлих, 61 зі 121

РОЗДІЛ 6
  хибно: CV від 1970-01-01 = 0.5949 %
  хибно: CV від 1900-01-01 = 0.2536 %
  хибно: CV від 2021-01-01 = 30.0414 %
  правильно: розмах 463 днів, IQR 152 днів, std 113.0 днів
  по місяцях: {Period('2022-03', 'M'): 18, Period('2022-04', 'M'): 30, Period('2022-05', 'M'): 11, Period('2022-06', 'M'): 1}

РОЗДІЛ 7
  хибно: Пірсон = 0.1767
помер/жив   0   1
Стать            
жінки      32  43
чоловіки   28  18
  правильно: chi2 = 3.0861, p = 0.0790, V Крамера = 0.1597

РОЗДІЛ 10
  хибно: кореляція «Дата розтину» з летальністю = 1.0
    заповнена у померлих: 61 | у живих: 0
  правильно, без ознак, що виникають після результату:
Лім2     0.645
ІЛ6-2    0.616
Л2       0.480
Вік      0.449
П/Я2     0.395


---
# Етап 7. Висновки та індивідуальне питання

In [22]:
ВИСНОВКИ = [
    'Стовпець «номер і/х» названо номером історії хвороби, але значення 1837 і 2526 трапляються '
    'двічі кожне, тому він не ідентифікує запис однозначно і для склеювання таблиць непридатний.',
    'Стовпець «Т вище38» містить ту саму величину у двох несумісних формах: 27 записів є голими '
    'числами, решта 50 — текстом із контекстом, через що pd.to_numeric мовчки втрачає дві третини '
    'спостережень і середнє падає з 4.221 до 3.444 дня.',
    'Для пари «Ліжко днів» і «Вік» Пірсон дає -0.0996 при p = 0.279, а Спірмен -0.1940 при '
    'p = 0.034: лінійної залежності немає, монотонна слабка є, і різниця пояснюється правим '
    'хвостом ліжко-днів, де максимум 79 при медіані 15.',
    'У двох хворих дата розтину стоїть раніше за дату поступлення, на 343 і 15 днів, причому в '
    'одному випадку рік розтину 2021 при виписці 2022 — тривалість від поступлення до розтину '
    'для них не визначена, тому ці два значення вилучено, а не підправлено.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')


1. Стовпець «номер і/х» названо номером історії хвороби, але значення 1837 і 2526 трапляються двічі кожне, тому він не ідентифікує запис однозначно і для склеювання таблиць непридатний.
2. Стовпець «Т вище38» містить ту саму величину у двох несумісних формах: 27 записів є голими числами, решта 50 — текстом із контекстом, через що pd.to_numeric мовчки втрачає дві третини спостережень і середнє падає з 4.221 до 3.444 дня.
3. Для пари «Ліжко днів» і «Вік» Пірсон дає -0.0996 при p = 0.279, а Спірмен -0.1940 при p = 0.034: лінійної залежності немає, монотонна слабка є, і різниця пояснюється правим хвостом ліжко-днів, де максимум 79 при медіані 15.
4. У двох хворих дата розтину стоїть раніше за дату поступлення, на 343 і 15 днів, причому в одному випадку рік розтину 2021 при виписці 2022 — тривалість від поступлення до розтину для них не визначена, тому ці два значення вилучено, а не підправлено.


In [23]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()


def is_identifier(s):
    s = s.dropna()
    if len(s) < 0.5 * len(df) or s.nunique() / len(s) < 0.95:
        return False
    return pd.to_numeric(s, errors='coerce').notna().mean() < 0.9


ids = [c for c in df.columns if is_identifier(df[c])]
print('стовпців у файлі:', len(df.columns))
print('розпізнано як ідентифікатори:', ids)

ВІДПОВІДЬ = (
    'Ідентифікаторів у файлі три: «номер і/х», «ПІП» і «Телефон». Розпізнати їх без назв можна '
    'за трьома ознаками одночасно. Перша — майже повна унікальність: у цих стовпцях частка '
    'унікальних значень від заповнених становить 0.983, 1.000 і 1.000, тоді як у вимірюваних '
    'ознак вона нижча, бо значення повторюються. Друга — покриття: ідентифікатор заповнений '
    'майже завжди, у нас 121, 121 і 120 рядків зі 121, тоді як «КТ опис» теж унікальний на всі '
    'сто відсотків, але заповнений лише у 10 хворих, тобто це рідкісний коментар, а не ключ. '
    'Третя — непридатність до арифметики: значення не зводяться до числа, бо містять роздільники '
    'й літери, як «2377/239» або «0958469788 дружина Ганна». Разом ці три умови дають рівно три '
    'стовпці й не чіпають лабораторних показників «ІЛ6-1», «Д-дим2» чи «ФЕРИТ1», які теж мають '
    'сто відсотків унікальності, але заповнені в 20, 30 і 2 хворих і є числами. Окремо треба '
    'сказати, що «номер і/х» ідентифікатором названо, а функцію його він не виконує: значення '
    '1837 і 2526 трапляються по два рази, тож ключем для склеювання таблиць цей стовпець бути '
    'не може. «Примітку» правило не зараховує до ідентифікаторів, бо вона заповнена лише в 45 '
    'хворих, але в модель її теж не беруть — з іншої причини, як вільний текст.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print()
print(ВІДПОВІДЬ)


ВАШЕ ПИТАННЯ:
Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?

стовпців у файлі: 77
розпізнано як ідентифікатори: ['номер і/х', 'ПІП', 'Телефон']

Ідентифікаторів у файлі три: «номер і/х», «ПІП» і «Телефон». Розпізнати їх без назв можна за трьома ознаками одночасно. Перша — майже повна унікальність: у цих стовпцях частка унікальних значень від заповнених становить 0.983, 1.000 і 1.000, тоді як у вимірюваних ознак вона нижча, бо значення повторюються. Друга — покриття: ідентифікатор заповнений майже завжди, у нас 121, 121 і 120 рядків зі 121, тоді як «КТ опис» теж унікальний на всі сто відсотків, але заповнений лише у 10 хворих, тобто це рідкісний коментар, а не ключ. Третя — непридатність до арифметики: значення не зводяться до числа, бо містять роздільники й літери, як «2377/239» або «0958469788 дружина Ганна». Разом ці три умови дають рівно три стовпці й не чіпають лабораторних показників «ІЛ6-1», «Д-дим2» чи «ФЕРИТ1», які теж мають сто

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [24]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Перерахував вручну коефіцієнт варіації дати розтину від двох різних нулів і переконався, '
    'що число змінюється з 0.0038 на 0.0016. Звірив, скільки значень стовпця «Т вище38» виживає '
    'після pd.to_numeric: 27 із 77, решта 50 записані словами. Перевірив кодування стовпця '
    '«помер/жив»: одиниця стоїть рівно в тих 61 хворого, у яких заповнена дата розтину. '
    'Прогнав правило пошуку ідентифікаторів і руками перевірив кожен із трьох знайдених '
    'стовпців, а також чому «КТ опис» та «ІЛ6-1» до них не потрапили.'
)
ПІДТВЕРДЖУЮ = True


In [25]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 6 готова до захисту.
Савков Олександр, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.